# Explainer examples

The explainer takes **review texts** (and optional review ids) and returns
`qualities`: structured shared qualities with quotes (JSON only).

Run from the **repo root** with `PYTHONPATH=.` and `ANTHROPIC_API_KEY` set in `.env`.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

ROOT = Path("..").resolve()
if not (ROOT / "explainer").exists():
    ROOT = Path(".").resolve()

load_dotenv(ROOT / ".env")
os.environ.setdefault("PYTHONPATH", str(ROOT))

from explainer import Explainer

## 1. Minimal call with inline review text

In [ ]:
seed_text = """
A restless, nocturnal record that trades hooks for texture. The production is
icy and meticulous, with fractured rhythms that keep the listener off-balance.
""".strip()

rec_text = """
Built from glitchy percussion and cold synth pads, this album favors atmosphere
over conventional song structure. Its rhythms feel deliberately unsettled.
""".strip()

explainer = Explainer()
result = explainer.explain(
    seed_text,
    rec_text,
    seed_review_id="demo-seed",
    rec_review_id="demo-rec",
    n=3,
)

for q in result.qualities:
    print(f"- {q.quality}")
    print(f"  A: {q.seed_quote}")
    print(f"  B: {q.rec_quote}")

## 2. Load two reviews from the cleaned dataset

Lookup stays outside the module — here the notebook finds reviews, then passes texts + ids.

In [ ]:
reviews_path = ROOT / "datasets" / "merged_dataset_cleaned.csv"
df = pd.read_csv(
    reviews_path,
    usecols=["review_id", "artist_norm", "album_norm", "cleaned_text", "dataset"],
)

def first_review(artist: str, album: str):
    hit = df[
        (df["artist_norm"] == artist.lower().strip())
        & (df["album_norm"] == album.lower().strip())
    ]
    if hit.empty:
        raise LookupError(f"No review for {artist} — {album}")
    row = hit.iloc[0]
    return str(row["cleaned_text"]), str(row["review_id"]), str(row["dataset"])

seed_text, seed_id, seed_src = first_review("radiohead", "kid a")
rec_text, rec_id, rec_src = first_review("muse", "showbiz")
print(seed_src, seed_id)
print(rec_src, rec_id)

In [ ]:
result = explainer.explain(
    seed_text,
    rec_text,
    seed_review_id=seed_id,
    rec_review_id=rec_id,
    n=3,
)

for q in result.qualities:
    print(f"\n{q.quality}")
    print(f"  seed: {q.seed_quote}")
    print(f"  rec:  {q.rec_quote}")